# 13.09 Ollama — 공급자 카탈로그

**`langchain-ollama`** 는 로컬 머신 또는 LAN 의 Ollama 데몬과 통신한다. **API 키 없이, 인터넷 단절 환경에서도** chat·embedding 을 모두 자체 처리한다.

## 학습 목표

- `ChatOllama` / `OllamaEmbeddings` 두 클래스로 끝나는 단순한 패키지 구조
- 모델 가져오기는 LangChain 이 아니라 `ollama pull <model>` CLI 가 담당
- 1순위 차별 기능: **완전 로컬 추론** — 데이터가 호스트 밖으로 나가지 않음
- 베이스 URL 변경으로 LAN 의 GPU 박스를 데몬으로 사용하는 패턴

## 13.09.1 환경 설정

필요 도구: **Ollama 데몬** (`brew install ollama` 또는 https://ollama.com/download). 모델은 `ollama pull qwen3.5:9b-q4_K_M` 등으로 미리 받아 둔다.
필요 패키지: `langchain-ollama`. API 키 없음.

```bash
uv pip install langchain-ollama
ollama pull qwen3.5:9b-q4_K_M    # 한 번만
```

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_ollama import ChatOllama

# probe — 데몬이 살아있는지 + 모델이 받혀 있는지 한 번에 확인
# qwen3.5 는 reasoning 모델이라 thinking 토큰을 먼저 쓴다 → num_predict 충분히
r = ChatOllama(model="qwen3.5:9b-q4_K_M", reasoning=False, num_predict=64).invoke("ping")
print(r.content[:120])

## 13.09.2 공급자 제품군 전체 지도

| 카테고리 | 심볼 | 카테고리 노트북 |
|---------|------|----------------|
| Chat | `ChatOllama` | [`01_chat_models/04_ollama.ipynb`](../01_chat_models/04_ollama.ipynb) |
| Embeddings | `OllamaEmbeddings` | [`02_embeddings/05_ollama.ipynb`](../02_embeddings/05_ollama.ipynb) |

Ollama 는 chat + embedding 두 클래스로 끝난다. retriever · loader · vector store 는 **다른 공급자 또는 로컬 라이브러리(FAISS/Chroma)** 와 짝지어 쓴다.

Ollama 가 다루는 모델군은 LangChain 측 코드가 아니라 **Ollama 데몬 측**에서 관리한다 — `ollama pull`, `ollama list`, `ollama rm` CLI.

## 13.09.3 기본 사용 — chat

최소 한 줄. `model=` 에는 `ollama list` 에 표시되는 정확한 ID 를 넣는다.

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:9b-q4_K_M", reasoning=False, num_predict=256)
print(model.invoke("한국어로 한 문장만: 너는 누구냐?").content)

## 13.09.4 공급자 특화 기능 — 완전 로컬 추론

Ollama 의 1순위 차별 기능은 **데이터가 호스트 밖으로 나가지 않는 추론**이다.

- 의료·금융·법률 등 **데이터 거버넌스가 강한** 도메인의 프로토타이핑
- **인터넷 단절** 환경 (현장 디바이스, 보안 망)
- API 비용 0, GPU/CPU 자원만 소비

튜닝 1순위 파라미터:

- `num_ctx` — 컨텍스트 길이. 모델별 한계 (예: qwen3.5 8k~32k). 늘리면 VRAM 사용량 급증
- `num_gpu` — GPU 에 올릴 레이어 수. 모자라면 CPU 로 fallback (느려짐)
- `temperature`, `top_k`, `top_p`, `num_predict` — 생성 제어
- `keep_alive` — 모델을 메모리에 유지하는 시간 (`"5m"`, `"-1"` 무한)

**LAN 의 GPU 박스 사용**: `base_url="http://gpu-host:11434"` 로 데몬을 가리키면 노트북 외부 호스트에서 추론. Ollama 데몬은 멀티 클라이언트를 큐로 처리한다.

In [ ]:
from langchain_ollama import ChatOllama

tuned = ChatOllama(
    model="qwen3.5:9b-q4_K_M",
    reasoning=False,        # qwen3.5 thinking 모드 끔 (간단한 응답 시)
    num_ctx=8192,           # 컨텍스트 길이
    temperature=0.2,        # 결정적 출력
    num_predict=256,        # 출력 토큰 한도
    keep_alive="5m",        # 5분간 메모리에 유지 → 다음 호출 cold start 회피
    # base_url='http://gpu-host:11434',  # LAN 의 GPU 박스를 가리킬 때만
)
print(tuned.invoke("한 줄로 자기소개").content)

## 13.09.5 가격 · 한도 · 모델 카탈로그

비용 모델은 단순하다 — **하드웨어 + 전기**. 클라우드 API 호출 0.

한도: **VRAM 이 한계**. 모델 크기·양자화·`num_ctx` 가 곱해진다. 부족하면 CPU 오프로드 → 토큰 속도 급락.

주요 모델 패밀리(2026-04 시점):

| 모델 ID | 파라미터 | VRAM (Q4) | 비고 |
|---------|----------|-----------|------|
| `qwen3.5:9b-q4_K_M` | 9B | ~6 GB | 일반 한국어/다국어, 가성비 |
| `qwen3.5:27b-q4_K_M` | 27B | ~17 GB | 추론 품질 한 단계 위 |
| `qwen3.5:35b-a3b-q4_K_M` | 36B(MoE 3B active) | ~23 GB | MoE 로 빠른 토큰/초 |
| `llama3.3:70b-q4_K_M` | 70B | ~40 GB | 라이선스 동의 필요 |
| `nomic-embed-text` | 137M | ~250 MB | 임베딩 768d |
| `mxbai-embed-large` | 335M | ~700 MB | 임베딩 1024d |

리전: **로컬** — 데이터 거주성이 곧 호스트 위치.

## 핵심 정리

- **chat 1순위**: 한국어/다국어 = `qwen3.5:9b-q4_K_M` (소형) ~ `qwen3.5:27b-q4_K_M` (중형). VRAM 여건에 맞춰
- **embedding 1순위**: 빠른 영문 = `nomic-embed-text`, 다국어/품질 = `mxbai-embed-large` 또는 HF `bge-m3` (Ollama 외부)
- **공급자 차별 1순위**: 인터넷·키 없이 완전 로컬 추론 → 거버넌스가 가장 큰 가치
- 모델 관리는 LangChain 이 아니라 **Ollama CLI** 의 책임 — `ollama pull/list/rm`

## 다음

- [`01_chat_models/04_ollama.ipynb`](../01_chat_models/04_ollama.ipynb) — ChatOllama 깊은 사용 (도구 호출, structured output)
- [`02_embeddings/05_ollama.ipynb`](../02_embeddings/05_ollama.ipynb) — OllamaEmbeddings + 로컬 vector store 조합
- [`13_providers/07_huggingface.ipynb`](07_huggingface.ipynb) — 비교: 또 다른 로컬/오픈모델 표준 (TGI/TEI)

## 참고

- `langchain-ollama` PyPI: https://pypi.org/project/langchain-ollama/
- Ollama 공식: https://ollama.com
- 모델 카탈로그: https://ollama.com/library